# 214. Agent Tool Error Recovery：怎样分类失败、幂等恢复并熔断？

> **面试问题：怎样区分 timeout、validation、permission、conflict 与业务拒绝，安全地查询 receipt/修复参数/刷新状态，并以预算和熔断避免重复副作用？**

## 先给结论

不要只背论文名或框架名。应当说明输入/状态合同、核心算法、失败分支、独立 oracle、指标和可回滚制品。以下均使用受控小数据验证实现机制；真实生产仍需替换模型、权限、索引、安全审计与线上评测。

## 一手资料

- [ReAct](https://arxiv.org/abs/2210.03629)
- [Toolformer](https://arxiv.org/abs/2302.04761)
- [AgentDojo](https://arxiv.org/abs/2406.13352)

In [ ]:
contract = {"mode": "controlled-demo", "oracle": "assertions", "production": "versioned"}  # 执行本行的状态、计算或校验逻辑。
assert contract["mode"] == "controlled-demo"  # 执行本行的状态、计算或校验逻辑。
assert contract["oracle"] == "assertions"  # 执行本行的状态、计算或校验逻辑。
assert contract["production"] == "versioned"  # 执行本行的状态、计算或校验逻辑。
assert len(contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 错误状态合同

工具错误必须包含 call id、code、是否已有副作用和版本。timeout、validation、permission、conflict、业务拒绝的修复不同；不能让 Agent 对所有 error 盲目重试。


In [ ]:
from dataclasses import dataclass  # 执行本行的状态、计算或校验逻辑。
@dataclass(frozen=True)  # 执行本行的状态、计算或校验逻辑。
class ToolResult:  # 执行本行的状态、计算或校验逻辑。
    call_id: str  # 执行本行的状态、计算或校验逻辑。
    code: str  # 执行本行的状态、计算或校验逻辑。
    side_effect: bool = False  # 执行本行的状态、计算或校验逻辑。
results = [ToolResult("c1", "timeout"), ToolResult("c2", "validation"), ToolResult("c3", "permission")]  # 执行本行的状态、计算或校验逻辑。
assert len(results) == 3  # 执行本行的状态、计算或校验逻辑。
assert results[0].code == "timeout"  # 执行本行的状态、计算或校验逻辑。
assert results[2].side_effect is False  # 执行本行的状态、计算或校验逻辑。


## 2. 分类到恢复动作

恢复器应是模型外的确定策略：timeout 先查 receipt，validation 修参数，permission 请求审批，conflict 刷新状态，business_denied 停止。未知 code fail closed，避免攻击者制造新错误类型绕过控制面。


In [ ]:
policy = {"timeout": "inspect_receipt", "validation": "repair_arguments", "permission": "request_approval", "conflict": "refresh_state", "business_denied": "stop"}  # 执行本行的状态、计算或校验逻辑。
def action(result):  # 执行本行的状态、计算或校验逻辑。
    return policy.get(result.code, "stop")  # 执行本行的状态、计算或校验逻辑。
assert action(results[0]) == "inspect_receipt"  # 执行本行的状态、计算或校验逻辑。
assert action(results[1]) == "repair_arguments"  # 执行本行的状态、计算或校验逻辑。
assert action(ToolResult("x", "unknown")) == "stop"  # 执行本行的状态、计算或校验逻辑。


## 3. timeout receipt

超时可能发生在服务已提交写操作之后。先按幂等 key 查询 receipt：completed 则复用结果，not_started 才可同 key 重试，unknown 必须升级人工/对账，不能猜测。


In [ ]:
ledger = {"c1": {"state": "completed", "result": "order-o1"}}  # 执行本行的状态、计算或校验逻辑。
def receipt(call_id):  # 执行本行的状态、计算或校验逻辑。
    return ledger.get(call_id, {"state": "unknown"})  # 执行本行的状态、计算或校验逻辑。
known = receipt("c1")  # 执行本行的状态、计算或校验逻辑。
assert known["state"] == "completed"  # 执行本行的状态、计算或校验逻辑。
assert known["result"] == "order-o1"  # 执行本行的状态、计算或校验逻辑。
assert receipt("missing")["state"] == "unknown"  # 执行本行的状态、计算或校验逻辑。


## 4. 参数修复谱系

validation 失败需要产生新 call 版本而非覆盖原调用。新参数仍要再次经过 schema/权限 gate，且 trace 中必须能从 repair call 回溯 parent call、错误信息和操作者。


In [ ]:
def repair(call_id, arguments, field, value):  # 执行本行的状态、计算或校验逻辑。
    updated = dict(arguments)  # 执行本行的状态、计算或校验逻辑。
    updated[field] = value  # 执行本行的状态、计算或校验逻辑。
    return {"call_id": call_id + "-r1", "parent": call_id, "arguments": updated}  # 执行本行的状态、计算或校验逻辑。
fixed = repair("c2", {"amount": -1}, "amount", 10)  # 执行本行的状态、计算或校验逻辑。
assert fixed["parent"] == "c2"  # 执行本行的状态、计算或校验逻辑。
assert fixed["arguments"]["amount"] == 10  # 执行本行的状态、计算或校验逻辑。
assert fixed["call_id"] != fixed["parent"]  # 执行本行的状态、计算或校验逻辑。


## 5. 预算与熔断

失败恢复也会消耗 token/工具预算。对同类错误或写工具设置尝试上限，到达阈值应 stop/degrade/escalate，而不是让模型无限思考；阈值本身应按风险类别评测。


In [ ]:
def allow_attempt(attempts, max_attempts, repeats, max_repeats):  # 执行本行的状态、计算或校验逻辑。
    return attempts < max_attempts and repeats < max_repeats  # 执行本行的状态、计算或校验逻辑。
assert allow_attempt(1, 3, 0, 2)  # 执行本行的状态、计算或校验逻辑。
assert not allow_attempt(3, 3, 0, 2)  # 执行本行的状态、计算或校验逻辑。
assert not allow_attempt(1, 3, 2, 2)  # 执行本行的状态、计算或校验逻辑。


## 6. 并发冲突

conflict 表示 plan 基于旧状态。必须重新读取版本化对象、标记旧观察 stale、重新计划，并让写工具检查 expected version；直接重试可能覆盖其他用户更新。


In [ ]:
def needs_refresh(result):  # 执行本行的状态、计算或校验逻辑。
    return action(result) == "refresh_state"  # 执行本行的状态、计算或校验逻辑。
conflict = ToolResult("c4", "conflict")  # 执行本行的状态、计算或校验逻辑。
assert needs_refresh(conflict)  # 执行本行的状态、计算或校验逻辑。
assert not needs_refresh(results[0])  # 执行本行的状态、计算或校验逻辑。
assert action(conflict) == "refresh_state"  # 执行本行的状态、计算或校验逻辑。


## 7. 恢复评测

只看最终任务成功会隐藏错误分类错、重复扣费和不必要重试。离线集应按 error code、读写、已提交状态和攻击输入切片，报告分类正确、恢复成功、重复副作用、成本和人工升级率。


In [ ]:
def metrics(traces):  # 执行本行的状态、计算或校验逻辑。
    return {"classified": sum(item["classified"] for item in traces) / len(traces), "recovered": sum(item["recovered"] for item in traces) / len(traces), "duplicate": sum(item["duplicate"] for item in traces) / len(traces)}  # 执行本行的状态、计算或校验逻辑。
report = metrics([{"classified": 1, "recovered": 1, "duplicate": 0}, {"classified": 1, "recovered": 0, "duplicate": 0}])  # 执行本行的状态、计算或校验逻辑。
assert report["classified"] == 1.0  # 执行本行的状态、计算或校验逻辑。
assert report["recovered"] == 0.5  # 执行本行的状态、计算或校验逻辑。
assert report["duplicate"] == 0.0  # 执行本行的状态、计算或校验逻辑。


## 8. 可复放制品

恢复 trace 应保存原 call、error code、policy version、receipt、repair 谱系、预算和终态。日志需保护敏感参数，并以结构化事件/哈希审计，而不是依赖模型自然语言解释。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"call": "c1", "code": "timeout", "action": action(results[0]), "receipt": known["state"], "policy": "recovery-v1"}  # 执行本行的状态、计算或校验逻辑。
fingerprint = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["action"] == "inspect_receipt"  # 执行本行的状态、计算或校验逻辑。
assert artifact["receipt"] == "completed"  # 执行本行的状态、计算或校验逻辑。
assert len(fingerprint) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

完整答案应先定义成功条件，再说明数据状态、主路径、失败边界和评测。受控断言只证明实现不变量，不能直接外推为真实大语料、模型语义、线上成本或安全效果。
